In [1]:
import os
from pathlib import Path

# ── data ──────────────────────────────────────────────────────────────────────
DATASET     = "soy"
SEED        = 42
SCRATCH     = os.environ.get("SVAR_SCRATCH", str(Path.home() / "svar_scratch"))

# ── embedding cache (for the two embedding paths) ─────────────────────────────
BACKBONE    = "carbon3b"                 # label only (cache is addressed by CACHE below)
HALF_WINDOW = 500
CACHE       = f"{SCRATCH}/caches/soy/carbon3b_hw500.ckpt.pt"   # or carbon500m_hw500.ckpt.pt
RECIPE      = "center_ln_mean"           # pooled-embedding recipe: center_ln_mean | sum_std

# ── classical model (paths 1 & 2) ────────────────────────────────────────────
MODEL       = "krr"                      # ridge | pls | svr | krr | rf | gbm
CV_FOLDS    = 5

# ── NN head (path 3) ──────────────────────────────────────────────────────────
HEAD        = "mlp"                      # mlp | linear
POOL        = "mean"                     # mean | sum
HIDDEN_DIM  = 512                        # None -> emb_dim
N_LAYERS    = 8
DROPOUT     = 0.25
LR          = 3e-4
EPOCHS      = 60
WEIGHT_DECAY= 1e-4
BATCH_SIZE  = 64

## Setup

In [2]:
import numpy as np
import torch

from training.common import features as feat, metrics as cmetrics
from training.common.datasets import get_dataset
from training.common.splits import get_or_build_split
from crop_embed.data.preprocessing import scale_phenotypes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def mean_pearson(m):
    """trait-mean Pearson from a cmetrics.evaluate() dict."""
    return m.get("mean", {}).get("pearson", float("nan"))

/home/andrew/anaconda3/envs/svar/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## Data — dataset spec, split, z-scored targets

One 70/15/15 split keyed by sample ID (seeded), identical across every model
family. Targets are per-trait z-scored (NaN-safe); Pearson/R² are invariant to
that, it just puts traits on a comparable scale.

In [3]:
spec   = get_dataset(DATASET)
split  = get_or_build_split(DATASET, seed=SEED)
samples    = split.sample_ids
trait_cols = split.trait_cols

Y = scale_phenotypes(split.targets).astype(np.float32)      # (n, T), z-scored, NaN=missing
tr = split.indices("train", samples)
va = split.indices("val",   samples)
te = split.indices("test",  samples)
print(f"{len(samples)} samples  |  {len(tr)} train / {len(va)} val / {len(te)} test")
print(f"{len(trait_cols)} traits: {trait_cols}")

14461 samples  |  10123 train / 2169 val / 2169 test
11 traits: ['protein', 'oil', 'Linoleic', 'Linolenic', 'R1', 'R8', 'Hgt', 'Ldg', 'SQ', 'SdWgt', 'Yield']


## Shared classical runner

Per-trait `GridSearchCV` (Pearson-scored, seeded KFold) inside train, select on
val, report test — exactly what `snp_sklearn`/`emb_sklearn` do. The estimator
(pipeline + grid) is the real `make_estimator`; `args` is a tiny namespace instead
of argparse.

## Path 2 — Embedding-classical

Pooled per-sample embedding (dense, low-dim), so no sparse/SVD. This is the
winning family: 3B-krr ≈ 0.70.

In [3]:
X_emb = feat.pooled_embeddings(spec, BACKBONE, HALF_WINDOW, samples,
                               recipe=RECIPE, cache_path=CACHE)
print("pooled embedding matrix:", X_emb.shape)
run_classical(X_emb, model=MODEL)

NameError: name 'spec' is not defined

## Path 3 — Embedding NN-head

The main head path from `training/emb_nn/run.py`, inlined and stripped to the
common case: load the window cache, pool to one vector per sample
(`embedding_bag`), warm-start a per-dim standardizer on train, train an
MLP/linear head with masked-MSE + AdamW (weight decay on linear weights only),
report val/test Pearson. (Dropped here vs. the runner: subtract-reference /
center-windows / SVD / bottleneck / early-stopping / wandb.)

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from crop_embed.models.fp_head_model import MLPModel, LinearModel, FPSumHeadModel
from crop_embed.train import masked_mse

torch.manual_seed(SEED)

# ── load cache, align targets to its sample order, pool to one vec/sample ──────
cached = feat.load_cached_windows(CACHE)
assert cached is not None, "legacy cache without sample_ids; rebuild via embed_windows.py"
cache = cached.cache.float()                      # (n_fingerprints, D)
sfi   = cached.sample_fp_index                    # (n_samples, n_windows) long
emb_dim, n_traits = cache.shape[1], len(trait_cols)

Yh = torch.tensor(scale_phenotypes(feat.load_targets(spec, cached.samples)[0]),
                  dtype=torch.float32)
tr_i = torch.tensor(split.indices("train", cached.samples))
va_i = torch.tensor(split.indices("val",   cached.samples))
te_i = torch.tensor(split.indices("test",  cached.samples))

summed = F.embedding_bag(sfi, cache, mode=POOL)   # (n_samples, D)
print(f"pooled: {tuple(summed.shape)}  ({POOL})  emb_dim={emb_dim}")

In [ ]:
# ── build head (warm-started, frozen standardizer) ────────────────────────────
inner = (LinearModel(emb_dim, n_traits) if HEAD == "linear"
         else MLPModel(emb_dim, n_traits, hidden_dim=HIDDEN_DIM,
                       n_layers=N_LAYERS, dropout=DROPOUT))
head = FPSumHeadModel(
    inner, emb_dim=emb_dim, normalize=True,
    warm_start_embeddings=summed[tr_i],           # fit standardizer on train
    pool=POOL, standardizer="perdim", freeze_standardizer=True,
).to(device)

# weight decay on inner Linear weights only
lin_ids = {id(m.weight) for m in head.modules() if isinstance(m, torch.nn.Linear)}
decay   = [p for p in head.parameters() if p.requires_grad and id(p) in lin_ids]
nodecay = [p for p in head.parameters() if p.requires_grad and id(p) not in lin_ids]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                         {"params": nodecay, "weight_decay": 0.0}], lr=LR)
print(f"head={HEAD} params={sum(p.numel() for p in head.parameters()):,}")

# ── train ─────────────────────────────────────────────────────────────────────
loader = DataLoader(TensorDataset(summed[tr_i], Yh[tr_i]),
                    batch_size=BATCH_SIZE, shuffle=True)
val_x, val_y = summed[va_i].to(device), Yh[va_i].to(device)
for epoch in range(EPOCHS):
    head.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = masked_mse(head.forward_postsum(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        head.eval()
        with torch.no_grad():
            vp = cmetrics.evaluate(val_y.cpu().numpy(),
                                   head.forward_postsum(val_x).cpu().numpy(), trait_cols)
        print(f"epoch {epoch:3d}  val_r={mean_pearson(vp):.4f}")

# ── final val/test ────────────────────────────────────────────────────────────
head.eval()
with torch.no_grad():
    vpred = head.forward_postsum(summed[va_i].to(device)).cpu().numpy()
    tpred = head.forward_postsum(summed[te_i].to(device)).cpu().numpy()
val_m  = cmetrics.evaluate(Yh[va_i].numpy(), vpred, trait_cols)
test_m = cmetrics.evaluate(Yh[te_i].numpy(), tpred, trait_cols)
print(f"\nNN head ({HEAD}, {N_LAYERS}L, drop={DROPOUT}, hid={HIDDEN_DIM}): "
      f"val_r={mean_pearson(val_m):.4f}  test_r={mean_pearson(test_m):.4f}")

## Compare everything logged so far

Every classical run (paths 1–2, and any `training.*.run` you launch) appends to
the shared manifest. NN-head runs from this notebook are **not** written there —
read their `test_m` above directly.

In [ ]:
from training.common.run_record import load_runs
df = load_runs()
soy = df[df.dataset == DATASET]
print(soy.groupby(["features", "model", "backbone"])["test.pearson"]
        .max().sort_values(ascending=False).to_string())